## Import lib

In [2]:

import pandas as pd 
import numpy as np 
import os
import sys
sys.path.append('../')
sys.path.append('../..')
import random

from sklearn.metrics import classification_report, confusion_matrix \
        , roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

from preprocessing.load_data import DataLoader

In [4]:
from detection.isolation_forest import IsolationForestDetector
from detection.lof import LocalOutlierFactorDetector
from detection.autoencoder import AutoencoderDetector

## Read data

In [5]:


card_data = pd.read_csv('../../data/raw/creditcard.csv')

print("Card data shape: ", card_data.shape) 

print(card_data.head(2))


data_use = DataLoader(data=card_data, label_col='Class')

data_use.remove_columns(['Time'])

Card data shape:  (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   

        V26       V27       V28  Amount  Class  
0 -0.189115  0.133558 -0.021053  149.62      0  
1  0.125895 -0.008983  0.014724    2.69      0  

[2 rows x 31 columns]


In [6]:

training, label = data_use.get_features(), data_use.get_labels()

print("Training data shape: ", training.shape) 

print(training.head(2))

print("label data shape: ", label.shape) 

print(label.head(2))

X_train, X_test, y_train, y_test = data_use.train_test_split(test_size=0.3, random_state=42)

print("X_train shape: ", X_train.shape) 
print("X_test shape: ", X_test.shape)

print("Sample X_train data: ", X_train.head(2))

Training data shape:  (284807, 29)
         V1        V2        V3        V4        V5        V6        V7  \
0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   

         V8        V9       V10  ...       V20       V21       V22       V23  \
0  0.098698  0.363787  0.090794  ...  0.251412 -0.018307  0.277838 -0.110474   
1  0.085102 -0.255425 -0.166974  ... -0.069083 -0.225775 -0.638672  0.101288   

        V24       V25       V26       V27       V28  Amount  
0  0.066928  0.128539 -0.189115  0.133558 -0.021053  149.62  
1 -0.339846  0.167170  0.125895 -0.008983  0.014724    2.69  

[2 rows x 29 columns]
label data shape:  (284807,)
0    0
1    0
Name: Class, dtype: int64
X_train shape:  (199364, 29)
X_test shape:  (85443, 29)
Sample X_train data:                V1        V2        V3        V4        V5        V6        V7  \
2557   -2.289565 -0.480260  0.818685 -1.706423  0.822102 -1.66

## Model Loading

In [7]:
lof_baseline_path = '../../models/checkpoint/baseline_lof/lof_baseline_model.pkl'
isolation_forest_baseline_path = '../../models/checkpoint/baseline_isolation_forest/isolation_forest_model.pkl'
autoencoder_baseline_path = '../../models/checkpoint/baseline_autoencoder/autoencoder_baseline_model.keras'

In [8]:
lof = LocalOutlierFactorDetector(n_neighbors=20, contamination=0.001)
lof.load_model(lof_baseline_path)

lof_predictions = lof.predict(X_test)
print(lof_predictions[:5])

print(classification_report(y_test, lof_predictions))

d:\thac_si_phenika\master_thesis\Opstimus\venv_opstimus\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


[ True False False False False]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     85307
           1       0.05      0.33      0.09       136

    accuracy                           0.99     85443
   macro avg       0.53      0.66      0.54     85443
weighted avg       1.00      0.99      0.99     85443



In [ ]:
isolation_forest = IsolationForestDetector(contamination=0.001, random_state=42)
isolation_forest.load_model(isolation_forest_baseline_path)

isolation_forest_predictions = isolation_forest.predict(X_test)
print(isolation_forest_predictions[:5])
print(classification_report(y_test, isolation_forest_predictions))

    print("Number of anomalies detected: ", np.sum(isolation_predictions))
    print("Percentage of anomalies detected: ", np.mean(isolation_predictions) * 100, "%")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, isolation_predictions.astype(int)))
    print("\nClassification Report:")
    print(classification_report(y_test, isolation_predictions.astype(int)))

    print("AUC-ROC of isolation forest: ", roc_auc_score(y_test, isolation_scores))

[ True False False False False]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     85307
           1       0.11      0.66      0.19       136

    accuracy                           0.99     85443
   macro avg       0.55      0.83      0.59     85443
weighted avg       1.00      0.99      0.99     85443

